In [1]:
import dataclasses
import numpy as np
import flax
print("Flax version", flax.__version__)
from flax import nnx
from typing import List
from s2ai.blocks.core_blocks import DiscoConvBlock, AverageBlock



class DiscoCNN(nnx.Module):
    """Disco convolutional network with spherical group normalisation."""

    def __init__(
        self,
        modes: List[str] = dataclasses.field(
            default_factory=lambda: ["disco", "disco", "disco", "disco"]
        ),
        rngs: nnx.Rngs = nnx.Rngs(0),
    ):
        # These could be nnx.vmapped/scanned for faster initialization
        self.disco_conv_block_0 = DiscoConvBlock(
            (128, 64), (1, 4), 2, mode=modes[0], rngs=rngs
        )
        self.disco_conv_block_1 = DiscoConvBlock(
            (64, 32), (4, 16), 4, mode=modes[1], rngs=rngs
        )
        self.disco_conv_block_2 = DiscoConvBlock(
            (32, 16), (16, 64), 8, mode=modes[2], rngs=rngs
        )
        self.disco_conv_block_3 = DiscoConvBlock(
            (16, 8), (64, 256), 16, mode=modes[3], rngs=rngs
        )

        self.average_block = AverageBlock(L_in=8)

        self.dropout_4 = nnx.Dropout(0.5, rngs=rngs)
        self.dense_4 = nnx.Linear(
            in_features=256,
            out_features=256,
            rngs=rngs,
        )
        self.activation_4 = nnx.swish

        self.dropout_5 = nnx.Dropout(0.5, rngs=rngs)
        self.dense_5 = nnx.Linear(
            in_features=256,
            out_features=10,
            rngs=rngs,
        )
        self.activation_5 = nnx.log_softmax

    def __call__(self, x):
        # Spherical convolutional blocks (L's, channels, groups)
        x = self.disco_conv_block_0(x)
        x = self.disco_conv_block_1(x)
        x = self.disco_conv_block_2(x)
        x = self.disco_conv_block_3(x)

        # Convert to equivariant dense layer
        x = self.average_block(x)

        # Standard MLP classifier
        x = self.dropout_4(x)
        x = self.dense_4(x)
        x = self.activation_4(x)
        x = self.dropout_5(x)
        x = self.dense_5(x)
        x = self.activation_5(x)
        return x



Flax version 0.10.2


JAX is not using 64-bit precision. This will dramatically affect numerical precision at even moderate L.


In [11]:
class DiscoCNNBuild(nnx.Module):
    "Builds spherical convolutional network based on set of tuples defining the convolution block per layer."
    def __init__(
        self,
        input_shape: tuple[int, ...],
        output_dim: int,
        layer_dims: list,
        activation: Callable = nnx.leaky_relu,
        rngs: nnx.Rngs | None = None, 
        ):

        if rngs is None:
            rngs = nnx.Rngs(0)

        self.input_shape = tuple(input_shape)
        self.output_dim = output_dim
        self.layer_dims = tuple(tuple(x) if not isinstance(x, tuple) else x for x in layer_dims)
        if len(self.layer_dims) == 0:
            raise ValueError("At least one layer dimension must be specified in layer_dims.")
        self.activation = activation
        self.rngs = rngs
        self.layers = nnx.List()
        
        raise NotImplementedError

    def check_layer_dims(
        self,
        input_shape: tuple[int, ...],
        output_dim: int,
        layer_dims: list,
        ):
        
        raise NotImplementedError

    def _add_layer(
        self,
        layer_dim: tuple,
        prev_layer_dim: tuple | None,
        layers: nnx.List,
        ):

        if prev_layer_dim is None:
            if len(layer_dim) == 2:
                if len(self.input_shape) == 1:
                    if self.input_shape[0] != layer_dim[0]:
                        raise ValueError(
                            f"Expected input shape {self.input_shape} to match linear "
                            f"input dimension {layer_dim[0]}."
                        )
                    layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))
                elif len(self.input_shape) == 3:
                    if prod(self.input_shape) != layer_dim[0]:
                        raise ValueError(
                            f"Expected flattened input size {prod(self.input_shape)} to "
                            f"match linear input dimension {layer_dim[0]}."
                        )
                    layers.append(self.Flatten())
                    layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))
                else:
                    raise ValueError(f"Unsupported input shape {self.input_shape}.")

            elif len(layer_dim) == 3:
                if len(self.input_shape) != 3:
                    raise ValueError(
                        f"Cannot add a convolutional layer after a linear layer, got "
                        f"layer dimensions {layer_dim} after input shape {self.input_shape}."
                    )
                if self.input_shape[2] != layer_dim[0]:
                    raise ValueError(
                        f"Expected input channels {self.input_shape[2]} to match "
                        f"convolutional input channels {layer_dim[0]}."
                    )
                layers.append(
                    nnx.Conv(
                        layer_dim[0],
                        layer_dim[1],
                        kernel_size=(layer_dim[2], layer_dim[2]),
                        padding="VALID",
                        rngs=self.rngs,
                    )
                )   

        else:
            if len(layer_dim) == 2 and len(prev_layer_dim) == 2:
                if prev_layer_dim[1] != layer_dim[0]:
                    raise ValueError(
                        f"Linear layer input dimension {layer_dim[0]} does not match "
                        f"previous layer output dimension {prev_layer_dim[1]}."
                    )
                layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))

            elif len(layer_dim) == 3 and len(prev_layer_dim) == 3:
                if prev_layer_dim[1] != layer_dim[0]:
                    raise ValueError(
                        f"Convolutional layer input channels {layer_dim[0]} do not match "
                        f"previous layer output channels {prev_layer_dim[1]}."
                    )
                layers.append(
                    nnx.Conv(
                        layer_dim[0],
                        layer_dim[1],
                        kernel_size=(layer_dim[2], layer_dim[2]),
                        padding="VALID",
                        rngs=self.rngs,
                    )
                )

            elif len(layer_dim) == 2 and len(prev_layer_dim) == 3:
                layers.append(self.Flatten())
                layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))

            elif len(layer_dim) == 3 and len(prev_layer_dim) == 2:
                raise ValueError(
                    f"Cannot add a convolutional layer after a linear layer, got "
                    f"layer dimensions {layer_dim} after {prev_layer_dim}."
                )

        raise NotImplementedError
    
    def __call__(self, x):
        if len(self.layers) > 1:
            for layer in self.layers[:-1]:
                x = layer(x)
                x = self.activation(x)
            x = self.layers[-1](x)
        elif len(self.layers) == 1:
            for layer in self.layers:
                x = layer(x)
        return x


NameError: name 'Callable' is not defined

In [2]:
directory = "/Users/brian/Downloads/map_compression_samples_5_tomo_bins_flm_1500_batch_1"
n_map = 1
for i in range(n_map):
    m = np.load(directory+f'/map_{i}.npy')
    print(m.shape)
p = np.load(directory+"/params.npy")
print(p.shape)

(22492500,)
(2000, 5)


In [ ]:
class In1500_Out5_DiscoCNN(nnx.Module):
    """Disco convolutional network with spherical group normalisation."""

    def __init__(
        self,
        modes: List[str] = dataclasses.field(
            default_factory=lambda: ["disco", "disco", "disco", "disco"]
        ),
        rngs: nnx.Rngs = nnx.Rngs(0),
    ):
        # These could be nnx.vmapped/scanned for faster initialization
        self.disco_conv_block_0 = DiscoConvBlock(
            (1500, 750), (5, 5), 2, mode=modes[0], rngs=rngs
        )
        self.disco_conv_block_1 = DiscoConvBlock(
            (750, 250), (5, 5), 4, mode=modes[1], rngs=rngs
        )
        self.disco_conv_block_2 = DiscoConvBlock(
            (250, 50), (5, 5), 8, mode=modes[2], rngs=rngs
        )
        self.disco_conv_block_3 = DiscoConvBlock(
            (50, 16), (5, 5), 16, mode=modes[3], rngs=rngs
        )

        self.average_block = AverageBlock(L_in=16)

        self.dropout_4 = nnx.Dropout(0.5, rngs=rngs)
        self.dense_4 = nnx.Linear(
            in_features=90,
            out_features=45,
            rngs=rngs,
        )
        self.activation_4 = nnx.swish

        self.dropout_5 = nnx.Dropout(0.5, rngs=rngs)
        self.dense_5 = nnx.Linear(
            in_features=45,
            out_features=5,
            rngs=rngs,
        )
        self.activation_5 = nnx.log_softmax

    def __call__(self, x):
        # Spherical convolutional blocks (L's, channels, groups)
        x = self.disco_conv_block_0(x)
        x = self.disco_conv_block_1(x)
        x = self.disco_conv_block_2(x)
        x = self.disco_conv_block_3(x)

        # Convert to equivariant dense layer
        x = self.average_block(x)

        # Standard MLP classifier
        x = self.dropout_4(x)
        x = self.dense_4(x)
        x = self.activation_4(x)
        x = self.dropout_5(x)
        x = self.dense_5(x)
        # x = self.activation_5(x)
        return x

s2_CNN = In1500_Out5_DiscoCNN()

from sbi_compression.methods.neural.flows import RQSplineFlow
import distrax

features_shape = (5,)
context_shape = (5,)
features_dim = prod(features_shape)
context_dim = prod(context_shape)
n_transforms = 4
n_bins = 8
range_min = -5
range_max = 5
bijector_type = distrax.RationalQuadraticSpline
conditioner_hidden_dims = ((32,32), (32,32))
flow = RQSplineFlow(
    features_dim, 
    context_dim, 
    n_transforms=self.n_transforms, 
    hidden_dims=self.conditioner_hidden_dims, 
    activation=getattr(nnx, self.activation), 
    n_bins=self.n_bins, 
    range_min=self.range_min, 
    range_max=self.range_max, 
    bijector_type=self.bijector_type
    )

class s2_AE_Flow(nnx.Module):

    def __init__(self,
        encoder,
        flow,
        encoder_mode: Literal['context', 'features'],
        ):
        self.encoder_mode = encoder_mode
        self.encoder = encoder
        self.flow = flow
    
    def __call__(self, x: Array, context: Array) -> Array:
        if self.encoder_mode == 'context':
            context_encoded = self.encoder(context)
            x = self.flow(x, context_encoded)
        elif self.encoder_mode == 'features':
            x_encoded = self.encoder(x)
            x = self.flow(x_encoded, context)
        return x
    
    def sample(self, num_samples: int, rng: Array, context: Array) -> Array:
        x = self.flow.sample(num_samples, rng, context)
        return x
        
    def encode(self, x: Array) -> Array:
        assert x.shape[-len(self.input_shape):] == self.input_shape # Make sure the input shape matches the encoder, assume batch first
        x = self.encoder(x)
        return x
    
    def mode(self, mode: Literal['train_encoder', 'train_flow', 'train_all', 'eval']) -> None:
        if mode == 'train_encoder':
            self.encoder.train()
            self.flow.eval()
        elif mode == 'train_flow':
            self.encoder.eval()
            self.flow.train()
        elif mode == 'train_all':
            self.encoder.train()
            self.flow.train()
        elif mode == 'eval':
            self.encoder.eval()
            self.flow.eval()
    
